# Analisis data empiris dan simulasi proses Poisson homogen

* **Mata Kuliah:** Pengantar Proses Stokastik
* **Dosen Pengampu:** Ade Susanti, S.Si., M.Si.
* **Topik:** Proses Poisson Homogen (waktu antar-kedatangan eksponensial, waktu tunggu Gamma, superposisi, dan penipisan/thinning)
* **Rujukan:** H. M. Taylor & S. Karlin (1998, Bab V); H. C. Tijms (2003, Bab 1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import sys
import os

# Impor modul verifikasi perhitungan
sys.path.append(os.path.abspath('../scripts'))
import verify_calculations as vc

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Modul komputasi dan pustaka berhasil diinisialisasi.")

## 1. Pembangkitan lintasan proses Poisson homogen

Waktu antar-kedatangan $T_n \sim \text{Eksp}(\lambda)$ saling bebas. Waktu kedatangan ke-$n$ dihitung melalui penjumlahan kumulatif $W_n = \sum_{i=1}^n T_i$.

In [ ]:
np.random.seed(42)
lambda_param = 5.0  # Laju rata-rata 5 kedatangan per satuan waktu
durasi_t = 20.0     # Periode observasi

# Bangkitkan waktu antar kedatangan eksponensial
n_estimasi = int(lambda_param * durasi_t * 1.5)
inter_arrivals = np.random.exponential(scale=1.0 / lambda_param, size=n_estimasi)
arrival_times = np.cumsum(inter_arrivals)
arrival_times = arrival_times[arrival_times <= durasi_t]
n_total = len(arrival_times)

print(f"Total kejadian teramati hingga t = {durasi_t}: {n_total} kejadian")
print(f"Estimasi laju empiris: {n_total / durasi_t:.4f} (Parameter sejati: {lambda_param})")

## 2. Visualisasi fungsi tangga $N(t)$ dan waktu tunggu Gamma

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Lintasan N(t)
t_steps = np.concatenate([[0], arrival_times])
n_steps = np.arange(len(t_steps))
ax1.step(t_steps, n_steps, where='post', color='#003366', linewidth=1.8, label='$N(t)$ Empiris')
ax1.plot([0, durasi_t], [0, lambda_param * durasi_t], '--', color='#cc0000', label=r'Garis Rata-rata $\mathbb{E}[N(t)] = \lambda t$')
ax1.set_title("Lintasan Proses Poisson Homogen $N(t)$", fontsize=11, fontweight='bold')
ax1.set_xlabel("Waktu ($t$)")
ax1.set_ylabel("Jumlah Kumulatif Kejadian $N(t)$")
ax1.legend()

# Subplot 2: Histogram Waktu Antar-Kedatangan vs Densitas Eksponensial
sample_inter = np.diff(np.concatenate([[0], arrival_times]))
count, bins, _ = ax2.hist(sample_inter, bins=15, density=True, alpha=0.6, color='#2ca02c', edgecolor='black', label='Data Empiris ($T_n$)')
x_vals = np.linspace(0, max(sample_inter), 200)
ax2.plot(x_vals, stats.expon.pdf(x_vals, scale=1.0 / lambda_param), 'r-', linewidth=2, label=r'Teoretis $\text{Eksp}(\lambda=5)$')
ax2.set_title("Distribusi Waktu Antar-Kedatangan $T_n$", fontsize=11, fontweight='bold')
ax2.set_xlabel("Waktu Antar-Kejadian ($t$)")
ax2.set_ylabel("Densitas Probabilitas")
ax2.legend()

plt.tight_layout()
plt.savefig('../figures/simulasi_poisson_homogen.png', dpi=300)
plt.show()

## 3. Uji indeks dispersi Poisson

Pada proses Poisson homogen, nilai harapan dan variansi bernilai sama: $\mathbb{E}[N(t)] = \text{Var}(N(t)) = \lambda t$. Rasio dispersi $D = s^2 / \bar{X}$ teoretis bernilai 1.

In [ ]:
# Partisi durasi observasi menjadi interval-interval berdurasi delta_t = 1 satuan waktu
delta_t = 1.0
n_intervals = int(durasi_t / delta_t)
counts_per_interval = []

for i in range(n_intervals):
    t_start = i * delta_t
    t_stop = (i + 1) * delta_t
    c = np.sum((arrival_times > t_start) & (arrival_times <= t_stop))
    counts_per_interval.append(c)

disp_test = vc.dispersion_index_test(counts_per_interval)
for k, v in disp_test.items():
    print(f"{k}: {v}")

## 4. Simulasi operasi superposisi dan penipisan (thinning)

* **Superposisi:** Penggabungan dua proses Poisson independen dengan laju $\lambda_1$ dan $\lambda_2$ menghasilkan proses Poisson dengan laju $\lambda_1 + \lambda_2$.
* **Penipisan (thinning):** Pemisahan acak dengan peluang $p$ dan $1-p$ menghasilkan dua proses Poisson independen dengan laju $p\lambda$ dan $(1-p)\lambda$.

In [ ]:
# Demonstrasi Thinning dengan p = 0.6 (Prioritas 1) dan 0.4 (Prioritas 2)
p_class1 = 0.6
labels = np.random.binomial(1, p_class1, size=len(arrival_times))

arrivals_class1 = arrival_times[labels == 1]
arrivals_class2 = arrival_times[labels == 0]

print(f"Laju Teoretis Sub-proses 1: {lambda_param * p_class1:.2f} | Laju Empiris: {len(arrivals_class1) / durasi_t:.2f}")
print(f"Laju Teoretis Sub-proses 2: {lambda_param * (1 - p_class1):.2f} | Laju Empiris: {len(arrivals_class2) / durasi_t:.2f}")

## 5. Analisis data empiris panggilan darurat Seattle 911 (23 Oktober 2026)

Data diunduh dari City of Seattle Open Data Portal (API Socrata: `kzjm-xkqj`), mencakup 314 panggilan darurat selama 24 jam pada 23 Oktober 2026.

In [ ]:
# Muat dataset empiris riil dan lakukan verifikasi
csv_file = "../data/raw/seattle_fire_911_calls_20261023.csv"
emp_results = vc.verify_empirical_dataset(csv_file)
print("HASIL VERIFIKASI DATA EMPIRIS SEATTLE 911:")
for k, v in emp_results.items():
    print(f"  {k:30s}: {v}")
